# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
!pip install -q duckdb pandas pyarrow huggingface_hub

In [16]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "HF_TOKEN not found.")

Token loaded successfully!


Connect to DuckDB

In [7]:
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

print("Extensions loaded!")

Extensions loaded!


In [9]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content item for one client on one report date. I will analyze the March 2026 partition (month = '2026-03') because it is a mid-panel month and avoids using the final month (June 2026), which is reserved as the test period.

In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
""").df()


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Category          | Fields                                                                                                                              |
| ----------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| **Features**      | `gsc_clicks`, `gsc_impressions`, `gsc_avg_position`, `ga4_sessions`, `ga4_engagement_rate` (only when `ga4_data_available IS TRUE`) |
| **Label / Proxy** | Future content performance (declining/not declining)                                                                                |
| **Context**       | `client_hash_id`, `content_hash_id`, `report_date`                                                                                  |
| **Excluded**      | IDs as model inputs, future information, June 2026 sample data, label-derived columns                                               |


gsc_clicks – Knowable at the decision moment because they are historical clicks.

gsc_impressions – Available before prediction because they are historical impressions.

gsc_avg_position – Historical search ranking available before prediction.

ga4_sessions – Used only when ga4_data_available IS TRUE.

ga4_engagement_rate – Historical engagement metric known before prediction.

In [18]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5;
""").df()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will use the March 2026 partition (month = 2026-03) because it is a mid-panel month. I will not use the June 2026 sample table for feature development or label design.

Query 1

In [23]:
con.sql(f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) c
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY 1,2,3
HAVING COUNT(*)>1
LIMIT 5;
""").df()





FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


Query 2 – Row count and date span

In [24]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


Query 3 – Availability (IS TRUE)

In [25]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
AND ga4_data_available IS TRUE;
""").df()



,available_rows
0,413966


## 4. Data limits

This dataset has unequal history across clients, and GA4 metrics are unavailable for some periods. The June 2026 sample is intentionally excluded because it represents the final month and could introduce leakage. Therefore, the results are directional and depend on the available historical data.

## Self-check



- [x] Every section above is filled.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words.
- [x] Committed to my repo under work/notebooks/.